# D2 - Instacart Market Basket Analysis: Luật kết hợp + Gom cụm

## 1. Nguồn - giấy phép - quy mô

| Mục | Thông tin |
|---|---|
| Dataset | Instacart Market Basket Analysis |
| Link | https://www.kaggle.com/datasets/psparks/instacart-market-basket-analysis |
| Đơn vị công bố | Instacart (qua Kaggle) |
| Giấy phép | CC BY-NC-SA 4.0 |
| Ngày tải | 2024 (dữ liệu cập nhật năm 2017) |
| Quy mô công bố | khoảng 3.4 triệu dòng trong `order_products`; nhiều bảng liên kết, hơn 34 thuộc tính |
| Kỹ thuật yêu cầu | **Luật kết hợp + Gom cụm** |

**Tri thức lĩnh vực:** mỗi đơn hàng gồm nhiều sản phẩm. Luật kết hợp giúp tìm các sản phẩm thường được mua cùng nhau; các đặc trưng tổng hợp theo khách hàng như số đơn, số sản phẩm và chu kỳ mua lại phù hợp để phân khúc khách hàng bằng gom cụm.

**Vị trí trong pipeline:** notebook này (Bài 1) chịu trách nhiệm khảo sát dữ liệu thô, kiểm tra thiếu/trùng/khóa liên kết, và tạo ra bảng chi tiết đơn hàng **đã làm sạch** (`data/processed/D2_instacart/order_details_clean.csv`). Bảng này là đầu vào dùng chung cho các notebook kỹ thuật phía sau:
- **Bài 3 — Luật kết hợp** (`luat-ket-hop.ipynb`): đọc trực tiếp `order_details_clean.csv`, sau đó thực hiện bước biến đổi *đặc thù* của luật kết hợp (gộp sản phẩm theo `aisle`, rời rạc hóa `order_hour_of_day`/`order_dow` thành item ngữ cảnh, định nghĩa giao dịch/giỏ) — bước này được giải trình trong chính notebook đó.
- **Bài 2 — Gom cụm** (mục 6 ở cuối notebook này): dùng đặc trưng tổng hợp theo khách hàng (`customer_features.csv`) suy ra từ cùng bảng đã làm sạch.


## 2. Từ điển dữ liệu

| Bảng/thuộc tính | Ý nghĩa | Kiểu | Thang đo |
|---|---|---|---|
| `orders.order_id` | Mã đơn hàng | int | định danh |
| `orders.user_id` | Mã khách hàng | int | định danh |
| `orders.order_number` | Thứ tự đơn của khách hàng | int | tỷ lệ |
| `orders.order_dow` | Thứ trong tuần đặt đơn (0-6) | int | danh nghĩa |
| `orders.order_hour_of_day` | Giờ trong ngày đặt đơn (0-23) | int | khoảng |
| `orders.days_since_prior_order` | Số ngày từ đơn trước | float | tỷ lệ |
| `order_products.product_id` | Mã sản phẩm | int | định danh |
| `order_products.add_to_cart_order` | Thứ tự thêm vào giỏ | int | thứ hạng |
| `order_products.reordered` | Sản phẩm có được mua lại | int | nhị phân |
| `products.product_name` | Tên sản phẩm | string | danh nghĩa |
| `products.aisle_id` | Mã nhóm hàng (aisle) | int | danh nghĩa |
| `aisles.aisle` | Tên nhóm hàng | string | danh nghĩa |

Ma trận kỹ thuật: **D2 = Luật kết hợp + Gom cụm**.

Ghi chú: `aisles` được nạp thêm ở notebook này (so với bản gốc) vì Bài 3 cần gộp sản phẩm về mức `aisle` để giảm số chiều one-hot và làm cơ sở rời rạc hóa — xem mục 4.


In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Tìm thư mục gốc của project
current = Path.cwd()

while current != current.parent:
    if (current / 'data' / 'raw' / 'D2_instacart').exists():
        break
    current = current.parent
ROOT = current
RAW = ROOT / 'data' / 'raw' / 'D2_instacart'

PROCESSED = ROOT / 'data' / 'processed' / 'D2_instacart'
PROCESSED.mkdir(parents=True, exist_ok=True)

OUT = ROOT / 'data' / 'processed' / 'D2_instacart' / 'outputs'
OUT.mkdir(parents=True, exist_ok=True)

SAMPLE_ROWS = 200000

def find_file(name):
    matches = list(RAW.rglob(name))

    if not matches:
        raise FileNotFoundError(
            f'Khong tim thay {name} trong {RAW}'
        )

    return matches[0]

# Doc du lieu tho

orders = pd.read_csv(find_file('orders.csv'))

order_products = pd.read_csv(
    find_file('order_products__prior.csv'),
    nrows=SAMPLE_ROWS
)

products = pd.read_csv(find_file('products.csv'))
aisles = pd.read_csv(find_file('aisles.csv'))

print(
    'orders:', orders.shape,
    '| order_products:', order_products.shape,
    '| products:', products.shape,
    '| aisles:', aisles.shape
)

print(
    f"\nGhi chu: chi doc {SAMPLE_ROWS:,} dong dau cua order_products__prior.csv "
    "(gioi han tai nguyen tinh toan khi chay Apriori/FP-Growth tren toan bo "
    "~3.2 trieu order). Cung mau nay se duoc dung xuyen suot Bai 1 va Bai 3."
)


orders: (3421083, 7) | order_products: (200000, 4) | products: (49688, 4) | aisles: (134, 2)

Ghi chu: chi doc 200,000 dong dau cua order_products__prior.csv (gioi han tai nguyen tinh toan khi chay Apriori/FP-Growth tren toan bo ~3.2 trieu order). Cung mau nay se duoc dung xuyen suot Bai 1 va Bai 3.


## 3. Khám phá dữ liệu và giá trị thiếu

Kiểm tra kiểu dữ liệu, khóa liên kết và tỷ lệ thiếu trước khi biến đổi. Các bản ghi thiếu `product_name`/`aisle` hoặc không khớp khóa liên kết (`order_id`, `product_id`, `aisle_id`) sẽ không được dùng ở bước làm sạch tiếp theo.


In [2]:
from IPython.display import display

missing = pd.concat({
    'orders': orders.isna().sum(),
    'order_products': order_products.isna().sum(),
    'products': products.isna().sum(),
    'aisles': aisles.isna().sum()
}, axis=1).fillna(0)
missing.to_csv(OUT / 'missing_report.csv', encoding='utf-8-sig')
missing['total_missing'] = missing.sum(axis=1)
display(missing[missing['total_missing'] > 0].sort_values('total_missing', ascending=False).head(20))

assert orders['order_id'].is_unique
assert products['product_id'].is_unique
assert aisles['aisle_id'].is_unique

# Kiem tra khoa lien ket: order_id / product_id trong order_products co khop voi cac bang tham chieu khong
orphan_orders = (~order_products['order_id'].isin(orders['order_id'])).sum()
orphan_products = (~order_products['product_id'].isin(products['product_id'])).sum()
orphan_aisles = (~products['aisle_id'].isin(aisles['aisle_id'])).sum()

print(f'order_products co order_id khong khop orders     : {orphan_orders:,}')
print(f'order_products co product_id khong khop products : {orphan_products:,}')
print(f'products co aisle_id khong khop aisles            : {orphan_aisles:,}')


,orders,order_products,products,aisles,total_missing
days_since_prior_order,206209.0,0.0,0.0,0.0,206209.0


order_products co order_id khong khop orders     : 0
order_products co product_id khong khop products : 0
products co aisle_id khong khop aisles            : 0


## 4. Làm sạch và hợp nhất dữ liệu chi tiết đơn hàng (dữ liệu dùng chung cho Bài 3)

Hợp nhất `order_products` với `orders`, `products` và `aisles` thành một bảng chi tiết ở mức **(order_id, product_id)**. Loại các bản ghi thiếu `product_name`/`aisle` (không khớp khóa liên kết) và loại trùng lặp `(order_id, product_id)` nếu có.

Đây là **dữ liệu processed dùng chung**, được lưu ra `data/processed/D2_instacart/order_details_clean.csv`. Từ bảng này:
- Bài 3 (luật kết hợp) sẽ đọc trực tiếp file này làm điểm xuất phát, rồi mới thực hiện bước biến đổi đặc thù của luật kết hợp (gộp theo `aisle`, rời rạc hóa `order_hour_of_day`/`order_dow` thành item ngữ cảnh) — **không đọc lại dữ liệu thô** và không lặp lại bước kiểm tra thiếu/khóa liên kết đã làm ở đây.
- Mục 5 bên dưới (đặc trưng khách hàng cho gom cụm) cũng dùng lại bảng này thay vì đọc `order_products__prior.csv` thêm 2 lần như bản trước.


In [3]:
# Hop nhat order_products + orders + products + aisles
order_details = (
    order_products
    .merge(
        orders[['order_id', 'user_id', 'order_number', 'order_dow',
                'order_hour_of_day', 'days_since_prior_order']],
        on='order_id', how='inner'
    )
    .merge(
        products[['product_id', 'product_name', 'aisle_id']],
        on='product_id', how='left'
    )
    .merge(aisles, on='aisle_id', how='left')
)

n_before = len(order_details)

# Loai ban ghi thieu product_name / aisle (khong khop khoa lien ket)
order_details = order_details.dropna(subset=['product_name', 'aisle'])

# Loai trung lap (order_id, product_id) neu co
order_details = order_details.drop_duplicates(subset=['order_id', 'product_id'])

order_details['product_name'] = order_details['product_name'].str.strip()
order_details['aisle'] = order_details['aisle'].str.strip()

n_after = len(order_details)
print(f'So dong truoc khi lam sach : {n_before:,}')
print(f'So dong sau khi lam sach   : {n_after:,}')
print(f'So dong bi loai            : {n_before - n_after:,} '
      f'({(n_before - n_after) / n_before:.2%})')

cols = ['order_id', 'user_id', 'order_number', 'order_dow', 'order_hour_of_day',
        'days_since_prior_order', 'product_id', 'product_name', 'aisle_id', 'aisle',
        'add_to_cart_order', 'reordered']
order_details = order_details[cols]

order_details.to_csv(PROCESSED / 'order_details_clean.csv', index=False, encoding='utf-8-sig')
print('Da luu:', PROCESSED / 'order_details_clean.csv')
display(order_details.head())


So dong truoc khi lam sach : 200,000
So dong sau khi lam sach   : 200,000
So dong bi loai            : 0 (0.00%)
Da luu: d:\data-mining\data\processed\D2_instacart\order_details_clean.csv


,order_id,user_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,product_id,product_name,aisle_id,aisle,add_to_cart_order,reordered
0,2,202279,3,5,9,8.0,33120,Organic Egg Whites,86,eggs,1,1
1,2,202279,3,5,9,8.0,28985,Michigan Organic Kale,83,fresh vegetables,2,1
2,2,202279,3,5,9,8.0,9327,Garlic Powder,104,spices seasonings,3,0
3,2,202279,3,5,9,8.0,45918,Coconut Butter,19,oils vinegars,4,1
4,2,202279,3,5,9,8.0,30035,Natural Sweetener,17,baking ingredients,5,0


## 5. Đặc trưng khách hàng cho gom cụm

Từ bảng `order_details` đã làm sạch ở mục 4, tổng hợp các đặc trưng hành vi theo `user_id`: số đơn, tổng số sản phẩm, số sản phẩm khác nhau, tỷ lệ mua lại, vị trí trung bình trong giỏ. Các đặc trưng này phục vụ cho bước gom cụm ở mục 6.


In [4]:
summary = (
    order_details.groupby('user_id')
    .agg(
        orders=('order_id', 'nunique'),
        products=('product_id', 'count'),
        unique_products=('product_id', 'nunique'),
        reorder_sum=('reordered', 'sum'),
        cart_sum=('add_to_cart_order', 'sum'),
        cart_count=('add_to_cart_order', 'count')
    )
    .reset_index()
)

customer_features = summary.assign(
    reorder_rate=summary['reorder_sum'] / summary['products'],
    mean_cart_position=summary['cart_sum'] / summary['cart_count']
).drop(columns=['reorder_sum', 'cart_sum', 'cart_count'])

customer_features.to_csv(PROCESSED / 'customer_features.csv', index=False, encoding='utf-8-sig')

numeric = ['orders', 'products', 'unique_products', 'reorder_rate', 'mean_cart_position']
z = np.abs((customer_features[numeric] - customer_features[numeric].mean()) / customer_features[numeric].std())
outliers = pd.DataFrame({'attribute': numeric, 'count_z_gt_3': (z > 3).sum().values})
outliers.to_csv(OUT / 'outlier_report.csv', index=False, encoding='utf-8-sig')
display(customer_features.head())


,user_id,orders,products,unique_products,reorder_rate,mean_cart_position
0,13,1,5,5,0.600000,3.0
1,23,1,9,9,0.000000,5.0
2,27,1,13,13,0.461538,7.0
3,36,1,3,3,0.333333,2.0
4,42,1,6,6,0.166667,3.5


## 6. Luật kết hợp — thực hiện đầy đủ ở Bài 3

Phần chuẩn bị giao dịch đặc thù (gộp sản phẩm theo `aisle`, rời rạc hóa `order_hour_of_day` và `order_dow` thành item ngữ cảnh, định nghĩa giao dịch = đơn hàng), thuật toán Apriori/FP-Growth, các độ đo support/confidence/lift và bước lọc luật được thực hiện đầy đủ trong notebook riêng **`luat-ket-hop.ipynb`** (Bài 3).

Notebook đó đọc trực tiếp `data/processed/D2_instacart/order_details_clean.csv` vừa tạo ở mục 4 làm điểm xuất phát, thay vì đọc lại dữ liệu thô hay lặp lại các bước khảo sát/làm sạch đã thực hiện ở notebook này. Việc tách riêng giúp tránh có hai phân tích luật kết hợp khác nhau (một bản đơn giản ở đây, một bản đầy đủ ở Bài 3) cho cùng một dữ liệu.


## 7. Gom cụm khách hàng và đánh giá

Chuẩn hóa đặc trưng trước K-Means. Chọn số cụm có silhouette cao trong khoảng thử nghiệm.


In [5]:
X = StandardScaler().fit_transform(customer_features[numeric].fillna(0))
scores = []
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)
    sample_idx = np.random.RandomState(42).choice(len(X), size=min(5000, len(X)), replace=False)
    scores.append({'k': k, 'inertia': model.inertia_, 'silhouette': silhouette_score(X[sample_idx], labels[sample_idx])})
scores = pd.DataFrame(scores)
scores.to_csv(OUT / 'clustering_scores.csv', index=False)
best_k = int(scores.loc[scores['silhouette'].idxmax(), 'k'])
customer_features['cluster'] = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X)
customer_features.to_csv(OUT / 'customer_clusters.csv', index=False, encoding='utf-8-sig')
display(scores)
display(customer_features.groupby('cluster')[numeric].mean().round(2))


,k,inertia,silhouette
0,2,56345.099664,0.434487
1,3,43779.949362,0.415154
2,4,32818.178253,0.394588
3,5,25330.586263,0.411209
4,6,21275.197143,0.413301
5,7,18727.499889,0.391264
6,8,16441.463843,0.382722


,orders,products,unique_products,reorder_rate,mean_cart_position
cluster,,,,,
0,1.05,7.00,6.96,0.59,3.91
1,1.25,22.89,22.13,0.59,10.44
